`from sklearn.linear_model import LogisticRegression` <br><br>

`model = LogisticRegression()`        # 1. create the model <br>
`model.fit(X_train, y_train)`         # 2. train it <br>
`predictions = model.predict(X_test)` # 3. make predictions <br>
`score = model.score(X_test, y_test)` # 4. evaluate it

# Day 1 — Preprocessing

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

np.random.seed(42)

df = pd.DataFrame({
    'voltage':     np.random.normal(3.7, 0.2, 100),
    'current':     np.random.normal(10.0, 1.5, 100),
    'temperature': np.random.normal(25.0, 3.0, 100),
    'fault':       np.random.randint(0, 2, 100)   # 0 = normal, 1 = fault
})

print(df.head())
print("Shape:", df.shape)

    voltage   current  temperature  fault
0  3.799343  7.876944    26.073362      0
1  3.672347  9.369032    26.682354      1
2  3.829538  9.485928    28.249154      1
3  4.004606  8.796584    28.161406      0
4  3.653169  9.758071    20.866992      0
Shape: (100, 4)


## Train/Test Split:

1. Training set — the model learns from this (typically 80%)
2. Test set — you evaluate the model on this (typically 20%)

In [14]:
from sklearn.model_selection import train_test_split

X = df[['voltage', 'current', 'temperature']].values   # features
y = df['fault'].values                                  # target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% goes to test set
    random_state=42     # same as np.random.seed — reproducibility
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

Training samples: 80
Test samples: 20


X is always features, the columns your model uses to make predictions. <br> 
y is always the target, what you're trying to predict. <br>
This naming convention is universal across all of ML.

## Feature Scaling
- Z-score normalization done automatically with `StandardScalar`

The critical rule: **fit only on training data, never on test data.** If you fit the scaler on the full dataset before splitting, information from the test set leaks into the training process. This is called **data leakage** and it's one of the most common ML mistakes.

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # learn mean/std AND transform
X_test_scaled  = scaler.transform(X_test)         # only transform, don't relearn

X_train, X_test = train_test_split(X)
scaler.fit_transform(X_train)     # only sees training data
scaler.transform(X_test)          # applies same scaling, no refitting

array([[ 1.18720992e+00,  2.08617011e+00, -1.51832939e+00],
       [-1.11691314e+00, -9.98322218e-01, -1.13052237e+00],
       [-8.76861575e-03, -5.44256989e-01,  3.84536745e-01],
       [ 4.46228675e-01, -1.07002965e+00,  7.08366102e-01],
       [ 1.79020739e+00,  2.07413046e+00,  3.40811002e-01],
       [ 3.42259527e-01, -1.46390261e+00,  6.63285763e-01],
       [ 1.49205938e-02, -1.89138697e+00,  3.31207442e-01],
       [-1.09662225e-01, -1.36413780e+00,  4.50336038e-01],
       [ 1.09489710e+00,  4.70352085e-01, -2.96515522e-01],
       [ 9.09565496e-01,  1.91380129e-01, -9.08911069e-01],
       [ 2.52242861e-01,  2.41949427e+00,  8.89341734e-01],
       [-9.71845451e-01,  4.70754810e-01, -7.74223666e-01],
       [-1.35629961e+00, -1.65798250e+00, -1.69282851e+00],
       [-2.07153897e-01, -1.48078599e+00,  6.60579753e-01],
       [ 9.85306165e-02, -1.54315030e-01,  9.60272400e-01],
       [ 1.41412801e-01, -7.29833892e-04,  6.25464805e-01],
       [-2.74658039e-01,  1.29770301e-01

## Encoding Categorical Variables

- ML models only understand numbers. If you have text columns like Compound (SOFT, MEDIUM, HARD) you need to convert them to numbers first.

### Label Encoding — converts each category to an integer:

In [17]:
# from sklearn.preprocessing import LabelEncoder

# le = LabelEncoder()
# df['Compound_encoded'] = le.fit_transform(df['Compound'])
# HARD → 0, MEDIUM → 1, SOFT → 2

### One-Hot Encoding — creates a separate binary column per category:

In [18]:
#from sklearn.preprocessing import OneHotEncoder
#import pandas as pd

#encoded = pd.get_dummies(df['Compound'], prefix='Compound')
#df = pd.concat([df, encoded], axis=1)

#    Compound_HARD  Compound_MEDIUM  Compound_SOFT
# 0              0                0              1
# 1              0                1              0
# 2              1                0              0

## Full Preprocessing Pipeline

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

# Sample dataset
df = pd.DataFrame({
    'LapTimeSeconds': [82.1, 83.4, 81.9, 84.2, 82.7],
    'TyreLife':       [3, 8, 1, 12, 5],
    'GridPosition':   [1, 4, 2, 8, 3],
    'Compound':       ['SOFT', 'MEDIUM', 'SOFT', 'HARD', 'SOFT'],
    'Strategy':       [0, 1, 0, 1, 0]   # 0=1-stop, 1=2-stop
})

# 1. Encode categorical column
df['Compound_encoded'] = LabelEncoder().fit_transform(df['Compound'])

# 2. Select features and target
X = df[['LapTimeSeconds', 'TyreLife', 'GridPosition', 'Compound_encoded']].values
y = df['Strategy'].values

# 3. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("X_train shape:", X_train_scaled.shape)
print("X_test shape:", X_test_scaled.shape)
print("Ready for a model")

X_train shape: (4, 4)
X_test shape: (1, 4)
Ready for a model
